# Maia2 personal fine-tune for `nick_p12`

Fine-tunes the Maia2 base **blitz** model on your own PGN games so it
plays your openings and roughly your strength.

**Repo:** [`nikhileshp/chess-clone`](https://github.com/nikhileshp/chess-clone)

**Before running:** Runtime → Change runtime type → GPU → A100 (or L4 / V100).

Two strategies in sequence:
* **Plan A** — use the package's built-in `maia2.train.run(cfg)` with a
  fine-tuning YAML (low LR, load pretrained, few epochs). Cleanest if
  the package exposes the right knobs.
* **Plan B** — if Plan A fails, fall back to a hand-written PyTorch loop
  on top of `model.from_pretrained()`.

## 1. Environment + GPU check

In [ ]:
!nvidia-smi

## 2. Clone the repo (code + compressed game data live here)

In [ ]:
%%bash
set -e
rm -rf /content/repo
git clone --depth 1 https://github.com/nikhileshp/chess-clone.git /content/repo
ls -lh /content/repo/data/clean/ || true
echo '---'
ls /content/repo/colab/

In [ ]:
import sys
sys.path.insert(0, '/content/repo')
sys.path.insert(0, '/content/repo/colab')
from finetune_helpers import FineTuneConfig, set_seed, log_environment, write_run_metadata, write_yaml_config
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
set_seed(42)
ENV = log_environment()

## 3. Install Maia2

PyTorch 2.4 + a few helpers. The package ships its own data loader that
reads `.pgn.zst` directly.

In [ ]:
# Note: maia2's setup.py is incomplete — it omits both python-chess and pyzstd
# as dependencies. We install them explicitly here. Without this, imports fail
# with `ModuleNotFoundError: No module named 'chess'` or `'pyzstd'`.
!pip install -q maia2 chess pyzstd zstandard pyyaml
!python -c 'import maia2, chess, pyzstd, torch; print("maia2:", getattr(maia2, "__version__", "?")); print("chess:", chess.__version__); print("pyzstd:", pyzstd.__version__); print("torch:", torch.__version__); print("cuda:", torch.cuda.is_available())'

## 4. Inspect the package surface

The Maia2 README gives a high-level outline but exact training/config
knobs aren't documented. We introspect the installed package so the
rest of this notebook can adapt to whatever the actual API is.

In [ ]:
import inspect
from maia2 import model, train, utils
from maia2 import inference, dataset

print('=== maia2.model exports ===')
print([n for n in dir(model) if not n.startswith('_')])
print()
print('=== maia2.train exports ===')
print([n for n in dir(train) if not n.startswith('_')])
print()
print('=== maia2.dataset exports ===')
print([n for n in dir(dataset) if not n.startswith('_')])
print()
print('=== model.from_pretrained signature ===')
print(inspect.signature(model.from_pretrained))
print()
if hasattr(train, 'run'):
    print('=== train.run signature ===')
    print(inspect.signature(train.run))
    print(inspect.getsource(train.run)[:1500])

## 5. Configure the fine-tune

In [ ]:
cfg = FineTuneConfig(
    user='nick_p12',
    pgn_zst_path='/content/repo/data/clean/all.pgn.zst',
    output_dir='/content/repo/colab_outputs',
    pretrained_type='blitz',
    epochs=3,
    learning_rate=1e-5,
    batch_size=256,
    val_fraction=0.05,
    freeze_encoder=False,
    weight_decay=1e-4,
    warmup_steps=200,
)
from pathlib import Path
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
meta_path = write_run_metadata(cfg, ENV, Path(cfg.output_dir))
yaml_path = write_yaml_config(cfg, Path(cfg.output_dir) / 'finetune.yaml')
print(open(yaml_path).read())

## 6. Plan A — try `maia2.train.run(cfg)` with our YAML

If this cell errors with a schema mismatch, read the error message and
either adjust `finetune_helpers.write_yaml_config` to match the package's
expected schema, OR skip to Plan B below.

In [ ]:
PLAN_A_OK = False
try:
    if hasattr(train, 'run'):
        train.run(str(yaml_path))
        PLAN_A_OK = True
    else:
        print('train.run() not available — falling through to Plan B')
except Exception as e:
    print(f'Plan A failed: {type(e).__name__}: {e}')
    print('Continue to Plan B (custom loop) below.')
print('Plan A succeeded:', PLAN_A_OK)

## 7. Plan B — custom PyTorch fine-tune loop

Only run these cells if Plan A failed. Loads the pretrained model
directly, builds a Dataset/DataLoader from your `.pgn.zst`, and trains a
few epochs of cross-entropy on your actual moves.

We import any data-prep helpers Maia2 exposes (e.g. `dataset.PGNDataset`)
rather than reinventing the position encoding.

In [ ]:
if not PLAN_A_OK:
    import torch
    from torch.utils.data import DataLoader
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('device:', device)
    
    # Load pretrained
    m = model.from_pretrained(type=cfg.pretrained_type, device='gpu' if device.type == 'cuda' else 'cpu')
    print('model class:', type(m).__name__)
    print('model param count:', sum(p.numel() for p in m.parameters()))
    print('model.forward signature:', inspect.signature(m.forward) if hasattr(m, 'forward') else 'n/a')
    
    # Inspect the dataset module to find a PGN-ingest class
    print('=== maia2.dataset attrs ===')
    for name in dir(dataset):
        if name.startswith('_'): continue
        obj = getattr(dataset, name)
        kind = type(obj).__name__
        print(f'  {name}: {kind}')

In [ ]:
# Plan B continued — finalize once we've confirmed model + dataset shapes above.
# This cell is intentionally left as a TODO that we fill in at runtime
# based on the introspection from the previous cell. Pseudocode:
#
#   ds = dataset.<PgnDatasetClass>(cfg.pgn_zst_path, ...)
#   train_size = int(len(ds) * (1 - cfg.val_fraction))
#   train_ds, val_ds = torch.utils.data.random_split(ds, [train_size, len(ds) - train_size])
#   train_dl = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=2)
#   val_dl = DataLoader(val_ds, batch_size=cfg.batch_size, num_workers=2)
#   opt = torch.optim.AdamW(m.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
#   sched = torch.optim.lr_scheduler.LinearLR(opt, start_factor=1e-3, total_iters=cfg.warmup_steps)
#   for epoch in range(cfg.epochs):
#       m.train()
#       for batch in train_dl:
#           # forward, compute policy CE on batch['move'], step
#           ...
#       # val pass: top-1 / top-3 move-prediction accuracy
#       ...
#   torch.save(m.state_dict(), f'{cfg.output_dir}/maia2_finetuned_{int(time.time())}.pt')
pass

## 8. Sanity check: does the fine-tuned model predict your moves?

Pick 50 random positions from your held-out games, ask the model for its
top-3 moves, and report top-1 / top-3 accuracy against the move you
actually played. Should be meaningfully higher than the base Maia2 blitz
model (the difference is the personalization signal).

In [ ]:
# Filled in once Plan A or Plan B has produced a checkpoint.
# Compares: base maia2-blitz vs fine-tuned, top-1 and top-3 accuracy on
# 50 random positions from a held-out subset of your games.
pass

## 9. Pull outputs back to your laptop

Either download via the file widget, or push to a `weights` branch of
the repo. The checkpoint is the only artifact you actually need.

In [ ]:
from google.colab import files
import glob
for p in sorted(glob.glob(f'{cfg.output_dir}/*.pt')):
    print('downloading', p)
    files.download(p)
for p in sorted(glob.glob(f'{cfg.output_dir}/*.json')):
    print('downloading', p)
    files.download(p)